In [2]:
import sys
print("Python executable:", sys.executable)

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import BertTokenizerFast, BertForSequenceClassification
from lime.lime_text import LimeTextExplainer
import shap
import matplotlib.pyplot as plt

print("All imports successful")
print("torch version:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Python executable: c:\Users\sehba\OneDrive - Asia Pacific University of Technology And Innovation (APU)\Documents\fraud-detector-fyp\backend\venv\Scripts\python.exe
All imports successful
torch version: 2.13.0+cpu
Device: cpu


In [3]:
MODEL_PATH = "../../ml/saved_models/bert_fraud_detector"
device = torch.device("cpu")

print("Loading tokenizer...")
tokenizer = BertTokenizerFast.from_pretrained(MODEL_PATH)

print("Loading model...")
model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

print("\nModel loaded successfully.")
print("Number of labels:", model.config.num_labels)
print("Model device:", next(model.parameters()).device)

# Quick sanity check — same test as Phase 3's local verification
test_text = "urgent hiring work from home send bank details immediately"
inputs = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=512,
    padding=True,
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits
probs = torch.softmax(logits, dim=1)

print(f"\nSanity check on known fraud text:")
print(f"  Genuine probability: {probs[0][0].item():.4f}")
print(f"  Fraud probability:   {probs[0][1].item():.4f}")
print(f"  Prediction: {'FRAUDULENT' if probs[0][1].item() >= 0.5 else 'GENUINE'}")

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Model loaded successfully.
Number of labels: 2
Model device: cpu

Sanity check on known fraud text:
  Genuine probability: 0.0003
  Fraud probability:   0.9997
  Prediction: FRAUDULENT


In [11]:
def predict_proba(texts):
    """
    Accepts a list of strings, numpy array, or single string.
    Returns numpy array of shape (n_texts, 2) with [genuine_prob, fraud_prob].
    Compatible with both LIME and SHAP.
    """
    # Handle all input types SHAP and LIME may pass
    if isinstance(texts, str):
        texts = [texts]
    elif isinstance(texts, np.ndarray):
        texts = texts.tolist()
        texts = [str(t) if not isinstance(t, str) else t for t in texts]

    # Filter out any non-string items that slipped through
    texts = [str(t) if not isinstance(t, str) else t for t in texts]

    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt",
    )
    encodings = {k: v.to(device) for k, v in encodings.items()}

    with torch.no_grad():
        logits = model(**encodings).logits

    probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs

# Quick verification
test_inputs = [
    "urgent hiring work from home send bank details immediately",
    "experienced marketing manager sought for leading uk firm",
]
result = predict_proba(test_inputs)
print("Output shape:", result.shape)
print("Row sums:", result.sum(axis=1))

# Also verify numpy array input works (what SHAP passes internally)
import numpy as np
np_input = np.array(test_inputs)
np_result = predict_proba(np_input)
print("Numpy input shape:", np_result.shape)
print("Wrapper verified for both LIME and SHAP.")

Output shape: (2, 2)
Row sums: [1. 1.]
Numpy input shape: (2, 2)
Wrapper verified for both LIME and SHAP.


In [5]:
test_df = pd.read_csv(
    "../../ml/data/processed/test.csv",
    index_col="original_index"
)

print("Test set loaded:", test_df.shape)
print("Fraudulent in test set:", test_df["fraudulent"].sum())

# Get model predictions on the full test set for categorisation
print("\nGenerating predictions on test set (this may take a few minutes on CPU)...")

all_probs = []
batch_size = 32
texts = test_df["clean_text"].tolist()
labels = test_df["fraudulent"].tolist()

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    batch_probs = predict_proba(batch)
    all_probs.append(batch_probs)
    if (i // batch_size) % 5 == 0:
        print(f"  Processed {min(i+batch_size, len(texts))}/{len(texts)} examples...")

all_probs = np.vstack(all_probs)
preds = (all_probs[:, 1] >= 0.5).astype(int)
labels_arr = np.array(labels)

# Categorise examples
tp_mask = (preds == 1) & (labels_arr == 1)  # fraud correctly caught
fn_mask = (preds == 0) & (labels_arr == 1)  # fraud missed
fp_mask = (preds == 1) & (labels_arr == 0)  # genuine incorrectly flagged
tn_mask = (preds == 0) & (labels_arr == 0)  # genuine correctly cleared

tp_indices = np.where(tp_mask)[0]
fn_indices = np.where(fn_mask)[0]
fp_indices = np.where(fp_mask)[0]
tn_indices = np.where(tn_mask)[0]

print(f"\nTest set prediction breakdown:")
print(f"  True Positives  (fraud caught):    {len(tp_indices)}")
print(f"  False Negatives (fraud missed):    {len(fn_indices)}")
print(f"  False Positives (genuine flagged): {len(fp_indices)}")
print(f"  True Negatives  (genuine correct): {len(tn_indices)}")
print(f"\nConfirmed: matches Phase 3 test evaluation exactly.")

# Select one example from each category for explanation
example_tp = test_df["clean_text"].iloc[tp_indices[0]]
example_fn = test_df["clean_text"].iloc[fn_indices[0]]
example_fp = test_df["clean_text"].iloc[fp_indices[0]]
example_tn = test_df["clean_text"].iloc[tn_indices[0]]

example_tp_title = test_df["title"].iloc[tp_indices[0]]
example_fn_title = test_df["title"].iloc[fn_indices[0]]
example_fp_title = test_df["title"].iloc[fp_indices[0]]
example_tn_title = test_df["title"].iloc[tn_indices[0]]

print(f"\nSelected examples for explanation:")
print(f"  True Positive:  '{example_tp_title}'")
print(f"  False Negative: '{example_fn_title}'")
print(f"  False Positive: '{example_fp_title}'")
print(f"  True Negative:  '{example_tn_title}'")

Test set loaded: (2682, 3)
Fraudulent in test set: 130

Generating predictions on test set (this may take a few minutes on CPU)...
  Processed 32/2682 examples...
  Processed 192/2682 examples...
  Processed 352/2682 examples...
  Processed 512/2682 examples...
  Processed 672/2682 examples...
  Processed 832/2682 examples...
  Processed 992/2682 examples...
  Processed 1152/2682 examples...
  Processed 1312/2682 examples...
  Processed 1472/2682 examples...
  Processed 1632/2682 examples...
  Processed 1792/2682 examples...
  Processed 1952/2682 examples...
  Processed 2112/2682 examples...
  Processed 2272/2682 examples...
  Processed 2432/2682 examples...
  Processed 2592/2682 examples...

Test set prediction breakdown:
  True Positives  (fraud caught):    111
  False Negatives (fraud missed):    19
  False Positives (genuine flagged): 13
  True Negatives  (genuine correct): 2539

Confirmed: matches Phase 3 test evaluation exactly.

Selected examples for explanation:
  True Positive

In [6]:
explainer = LimeTextExplainer(
    class_names=["genuine", "fraudulent"],
    random_state=42,
)

print("Generating LIME explanation for True Positive example...")
print(f"Job title: '{example_tp_title}'")
print(f"Text preview: {example_tp[:200]}...")
print()

tp_prob = predict_proba([example_tp])[0]
print(f"Model prediction: Genuine={tp_prob[0]:.4f} | Fraud={tp_prob[1]:.4f}")
print()

# num_features=15: show top 15 contributing words
# num_samples=500: generate 500 masked variants to fit the local linear model
# Fewer samples = faster but less stable; 500 is a good balance for CPU
exp_tp = explainer.explain_instance(
    example_tp,
    predict_proba,
    num_features=15,
    num_samples=500,
    labels=[1],
)

print("Top 15 words contributing to FRAUDULENT prediction:")
print(f"{'Word':<20} {'Weight':>10}  {'Direction'}")
print("-" * 45)
for word, weight in exp_tp.as_list(label=1):
    direction = "→ FRAUD" if weight > 0 else "→ GENUINE"
    print(f"{word:<20} {weight:>10.4f}  {direction}")

Generating LIME explanation for True Positive example...
Job title: 'IC&E Technician'
Text preview: ic e technician staffing recruiting done right for the oil energy industry represented candidates are automatically granted the following perks expert negotiations on your behalf maximizing your compe...

Model prediction: Genuine=0.0001 | Fraud=0.9999

Top 15 words contributing to FRAUDULENT prediction:
Word                     Weight  Direction
---------------------------------------------
basics                   0.0095  → FRAUD
times                    0.0079  → FRAUD
designs                  0.0075  → FRAUD
referral                 0.0070  → FRAUD
implimenting             0.0058  → FRAUD
rotating                -0.0054  → GENUINE
write                    0.0050  → FRAUD
background               0.0042  → FRAUD
four                    -0.0031  → GENUINE
driver                   0.0031  → FRAUD
assignments             -0.0025  → GENUINE
monitor                 -0.0018  → GENUINE
coord

In [7]:
print("Generating LIME explanation for False Negative example...")
print(f"Job title: '{example_fn_title}'")
print(f"Text preview: {example_fn[:200]}...")
print()

fn_prob = predict_proba([example_fn])[0]
print(f"Model prediction: Genuine={fn_prob[0]:.4f} | Fraud={fn_prob[1]:.4f}")
print()

exp_fn = explainer.explain_instance(
    example_fn,
    predict_proba,
    num_features=15,
    num_samples=500,
    labels=[1],
)

print("Top 15 words contributing to prediction (False Negative — fraud MISSED):")
print(f"{'Word':<20} {'Weight':>10}  {'Direction'}")
print("-" * 45)
for word, weight in exp_fn.as_list(label=1):
    direction = "→ FRAUD" if weight > 0 else "→ GENUINE"
    print(f"{word:<20} {weight:>10.4f}  {direction}")

print()
print("NOTE: This posting IS fraudulent but was classified as genuine.")
print("Words pushing toward GENUINE (negative weights) explain why the model missed it.")

Generating LIME explanation for False Negative example...
Job title: 'Customer Service'
Text preview: customer service smart creative personable customer service employee wanted responsibilities include answering phones entering and tracking customer orders light bookkeeping some running around qualif...

Model prediction: Genuine=0.9967 | Fraud=0.0033

Top 15 words contributing to prediction (False Negative — fraud MISSED):
Word                     Weight  Direction
---------------------------------------------
creative                -0.5273  → GENUINE
smart                   -0.2723  → GENUINE
customer                 0.1892  → FRAUD
responsibilities        -0.1463  → GENUINE
qualifications          -0.0890  → GENUINE
include                 -0.0760  → GENUINE
position                 0.0755  → FRAUD
running                  0.0752  → FRAUD
some                     0.0740  → FRAUD
answering                0.0693  → FRAUD
phones                   0.0650  → FRAUD
employee             

In [8]:
print("Generating LIME explanation for False Positive example...")
print(f"Job title: '{example_fp_title}'")
fp_prob = predict_proba([example_fp])[0]
print(f"Model prediction: Genuine={fp_prob[0]:.4f} | Fraud={fp_prob[1]:.4f}")
print()

exp_fp = explainer.explain_instance(
    example_fp,
    predict_proba,
    num_features=15,
    num_samples=500,
    labels=[1],
)

print("Top 15 words (False Positive — genuine posting incorrectly flagged as fraud):")
print(f"{'Word':<20} {'Weight':>10}  {'Direction'}")
print("-" * 45)
for word, weight in exp_fp.as_list(label=1):
    direction = "→ FRAUD" if weight > 0 else "→ GENUINE"
    print(f"{word:<20} {weight:>10.4f}  {direction}")

print()
print("="*50)
print()

print("Generating LIME explanation for True Negative example...")
print(f"Job title: '{example_tn_title}'")
tn_prob = predict_proba([example_tn])[0]
print(f"Model prediction: Genuine={tn_prob[0]:.4f} | Fraud={tn_prob[1]:.4f}")
print()

exp_tn = explainer.explain_instance(
    example_tn,
    predict_proba,
    num_features=15,
    num_samples=500,
    labels=[1],
)

print("Top 15 words (True Negative — genuine posting correctly cleared):")
print(f"{'Word':<20} {'Weight':>10}  {'Direction'}")
print("-" * 45)
for word, weight in exp_tn.as_list(label=1):
    direction = "→ FRAUD" if weight > 0 else "→ GENUINE"
    print(f"{word:<20} {weight:>10.4f}  {direction}")

Generating LIME explanation for False Positive example...
Job title: 'Project Accountant Asst'
Model prediction: Genuine=0.2111 | Fraud=0.7889

Top 15 words (False Positive — genuine posting incorrectly flagged as fraud):
Word                     Weight  Direction
---------------------------------------------
asst                     0.5301  → FRAUD
project                 -0.2865  → GENUINE
hs                      -0.2662  → GENUINE
forecast                -0.1776  → GENUINE
conducts                -0.1444  → GENUINE
analyzes                -0.1325  → GENUINE
to                      -0.1306  → GENUINE
a                        0.1302  → FRAUD
accountant               0.1141  → FRAUD
ability                  0.1005  → FRAUD
dollars                  0.0957  → FRAUD
level                   -0.0941  → GENUINE
income                   0.0871  → FRAUD
these                    0.0838  → FRAUD
diploma                  0.0828  → FRAUD


Generating LIME explanation for True Negative example...
J

In [12]:
print("Setting up SHAP explainer...")
print("Note: SHAP runs multiple forward passes per example - expect 1-2 min per example on CPU")
print()

# Use a small masker with a short max_evals to keep it tractable on CPU
masker = shap.maskers.Text(tokenizer)

shap_explainer = shap.Explainer(
    predict_proba,
    masker,
    output_names=["genuine", "fraudulent"],
)

# Select 3 true positive examples for SHAP analysis
# (confirmed fraud — most meaningful for global fraud pattern analysis)
tp_texts = [test_df["clean_text"].iloc[idx] for idx in tp_indices[:3]]
tp_titles = [test_df["title"].iloc[idx] for idx in tp_indices[:3]]

print(f"Running SHAP on {len(tp_texts)} true positive examples:")
for i, title in enumerate(tp_titles):
    print(f"  {i+1}. {title}")
print()
print("Starting SHAP computation (this will take several minutes)...")

shap_values = shap_explainer(tp_texts)

print("SHAP computation complete.")
print("shap_values shape:", shap_values.shape)
print("Output classes:", shap_values.output_names)

Setting up SHAP explainer...
Note: SHAP runs multiple forward passes per example - expect 1-2 min per example on CPU

Running SHAP on 3 true positive examples:
  1. IC&E Technician
  2. Payroll Data Coordinator Positions - Earn $100-$200 Daily 
  3. franciscan st. francis health

Starting SHAP computation (this will take several minutes)...


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:  33%|███▎      | 1/3 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 100%|██████████| 3/3 [07:39<00:00, 62.51s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 4it [10:14, 204.93s/it]                      

SHAP computation complete.
shap_values shape: (3, None, 2)
Output classes: [np.str_('genuine'), np.str_('fraudulent')]


In [ ]:
print("SHAP Token Attribution — Example 1 (IC&E Technician)")
print("Positive values = push toward FRAUD | Negative = push toward GENUINE")
print()

# Extract SHAP values for the fraudulent class (index 1) for first example
example_idx = 0
sv = shap_values[example_idx]

# Get tokens and their fraud-class SHAP values
tokens = sv.data
fraud_shap = sv.values[:, 1]

# Show top 15 tokens by absolute SHAP value
token_shap_pairs = list(zip(tokens, fraud_shap))
token_shap_pairs_sorted = sorted(
    token_shap_pairs,
    key=lambda x: abs(x[1]),
    reverse=True
)[:15]

print(f"{'Token':<20} {'SHAP Value':>12}  {'Direction'}")
print("-" * 48)
for token, sv_val in token_shap_pairs_sorted:
    direction = "→ FRAUD" if sv_val > 0 else "→ GENUINE"
    print(f"{str(token):<20} {sv_val:>12.4f}  {direction}")

print()

# Global summary: aggregate top fraud tokens across all 3 examples
print("="*50)
print("GLOBAL: Top fraud-driving tokens across all 3 examples")
print("="*50)

from collections import defaultdict
token_shap_global = defaultdict(list)

for i in range(len(shap_values)):
    sv_i = shap_values[i]
    for token, sv_val in zip(sv_i.data, sv_i.values[:, 1]):
        token_str = str(token).strip()
        if len(token_str) > 2:  # skip very short tokens like punctuation
            token_shap_global[token_str].append(sv_val)

# Average SHAP value per token across examples it appeared in
token_avg_shap = {
    token: np.mean(vals)
    for token, vals in token_shap_global.items()
    if len(vals) >= 1
}

# Top 20 fraud-driving tokens (highest positive average SHAP)
top_fraud_tokens = sorted(
    token_avg_shap.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]

print(f"\n{'Token':<20} {'Avg SHAP':>12}  {'Direction'}")
print("-" * 48)
for token, avg_val in top_fraud_tokens:
    direction = "→ FRAUD" if avg_val > 0 else "→ GENUINE"
    print(f"{token:<20} {avg_val:>12.4f}  {direction}")

# Bar chart of top fraud-driving tokens
tokens_plot = [t for t, _ in top_fraud_tokens[:15]]
values_plot = [v for _, v in top_fraud_tokens[:15]]
colors = ["#d73027" if v > 0 else "#4575b4" for v in values_plot]

plt.figure(figsize=(10, 6))
plt.barh(range(len(tokens_plot)), values_plot, color=colors)
plt.yticks(range(len(tokens_plot)), tokens_plot)
plt.xlabel("Average SHAP Value (positive = fraud signal)")
plt.title("Top 15 Fraud-Driving Tokens (SHAP, across 3 TP examples)")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig("../../ml/saved_models/shap_global_summary.png", dpi=150)
plt.show()
print("\nChart saved to ml/saved_models/shap_global_summary.png")